<a href="https://colab.research.google.com/github/Devanshujain336/Text_summariser/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd

df = pd.read_csv(
    '/content/drive/MyDrive/samsum-train.csv'
)

df.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [3]:
!pip install "transformers[torch]"

In [4]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

In [5]:
train_data = pd.read_csv(
    '/content/drive/MyDrive/samsum-train.csv'
)

val_data = pd.read_csv('/content/drive/MyDrive/samsum-validation.csv')


In [6]:
val_data

,id,dialogue,summary
0,13817023,"A: Hi Tom, are you busy tomorrow’s afternoon?\...",A will go to the animal shelter tomorrow to ge...
1,13716628,Emma: I’ve just fallen in love with this adven...,Emma and Rob love the advent calendar. Lauren ...
2,13829420,Jackie: Madison is pregnant\r\nJackie: but she...,Madison is pregnant but she doesn't want to ta...
3,13819648,Marla: <file_photo>\r\nMarla: look what I foun...,Marla found a pair of boxers under her bed.
4,13728448,Robert: Hey give me the address of this music ...,Robert wants Fred to send him the address of t...
...,...,...,...
813,13829423,Carla: I've got it...\r\nDiego: what?\r\nCarla...,Carla's date for graduation is on June 4th. Di...
814,13727710,"Gita: Hello, this is Beti's Mum Gita, I wanted...",Bev is going on the school trip with her son. ...
815,13829261,"Julia: Greg just texted me\r\nRobert: ugh, del...",Greg cheated on Julia. He apologises to her. R...
816,13680226,"Marry: I broke my nail ;(\r\nTina: oh, no!\r\n...",Marry broke her nail and has a party tomorrow....


In [7]:
train_data.shape

(14732, 3)

In [8]:
val_data.shape

(818, 3)

In [9]:
#random sampling

train_data = train_data.sample(n=4000, random_state=42).reset_index(drop=True)
val_data = val_data.sample(n=500,random_state=42).reset_index(drop=True)

In [10]:
train_data.shape

(4000, 3)

# Data Preprocessing

In [11]:
import re

def clean_data(text):
  text = re.sub(r"\r\n", " ", text) #remove lines
  text = re.sub(r"\s+", " ", text) #remove spaces
  text = re.sub(r"<.*?>", " ", text) # html tags
  text = text.strip().lower()
  return text



In [12]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)


val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

In [13]:
# Tokenisation

In [14]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [15]:
# raw data == > tokenized inputs for fine-tuning

def tokenize(data):
  inputs = tokenizer(data["dialogue"], padding="max_length", max_length=512, truncation=True)
  targets = tokenizer(data["summary"], padding="max_length", max_length=150, truncation=True)

  inputs["labels"] = targets["input_ids"]

  return inputs

In [16]:
train_dataset = train_data.apply(tokenize,axis=1).tolist()
val_dataset = val_data.apply(tokenize,axis=1).tolist()

In [17]:
import os
# Display the absolute path of the local results directory and saved model directory
print("Absolute path for training results:", os.path.abspath("./result"))
print("Absolute path for saved model:", os.path.abspath("./saved_summary_model"))

# Let's see what is inside the current workspace
print("\nContents of the current directory:")
for item in os.listdir('.'):
    print(f"- {item}")

Absolute path for training results: /content/result
Absolute path for saved model: /content/saved_summary_model

Contents of the current directory:
- .config
- drive
- sample_data


Working with the model : Fine tune the model


In [18]:
# NLP and generational Task --> GenAi based project

model = T5ForConditionalGeneration.from_pretrained("t5-small")

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [19]:
import torch

# Train over GPU if available, else fallback to CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Using device:", device)

Using device: cuda


In [20]:
# Training arguments pointing directly to Google Drive

training_args = TrainingArguments(
    output_dir = "/content/drive/MyDrive/Colab_Output/results",

    num_train_epochs = 6,
    weight_decay = 0.01,

    per_device_train_batch_size = 8,
    per_device_eval_batch_size = 8,

    eval_strategy = "epoch",
    save_strategy = "epoch",

    warmup_steps = 500
)

In [21]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset

)

In [22]:

trainer.train()

Epoch,Training Loss,Validation Loss
1,3.625844,0.381508
2,0.397262,0.359899
3,0.373885,0.354667
4,0.362884,0.350354
5,0.355325,0.350455
6,0.351533,0.349717


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=0.9111224568684896, metrics={'train_runtime': 1372.8436, 'train_samples_per_second': 17.482, 'train_steps_per_second': 2.185, 'total_flos': 3248203235328000.0, 'train_loss': 0.9111224568684896, 'epoch': 6.0})

In [23]:
# model load => fine tune => save the model

In [24]:
# Save final trained model directly to Google Drive
model.save_pretrained("/content/drive/MyDrive/Colab_Output/saved_summary_model")
tokenizer.save_pretrained("/content/drive/MyDrive/Colab_Output/saved_summary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/Colab_Output/saved_summary_model/tokenizer_config.json',
 '/content/drive/MyDrive/Colab_Output/saved_summary_model/tokenizer.json')

In [25]:
model = T5ForConditionalGeneration.from_pretrained("/content/drive/MyDrive/Colab_Output/saved_summary_model")
tokenizer = T5Tokenizer.from_pretrained("/content/drive/MyDrive/Colab_Output/saved_summary_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Test the core logic for summarisation

In [26]:
def summarise_dialogue(dialogue):
  dialogue = clean_data(dialogue) #clean

  #tokenize
  inputs = tokenizer(
      dialogue,
      padding = "max_length",
      max_length = 512,
      truncation = True,
      return_tensors = "pt"
  )

  #generate the summary => Token ids

  model.to(device)

  targets = model.generate(
      inputs_ids = inputs["input_ids"],
      attention_mask=input["attention_mask"],
      max_length=150,
      num_beams = 4,
      early_stopping = True
  )



  # token - ids => text summary (decoding)

  summary = tokenizer.decode(targets[0], skip_special_tokens=True)

  return summary


